In [ ]:
import json
import os

import whisper


# ── Configuration ──────────────────────────────────────────────────────────────
AUDIO_DIR = "audios"   # folder that contains all .mp3 files
OUTPUT_DIR = "jsons"   # raw Whisper output (one JSON per audio file)

MODEL_NAME = "base"    # Whisper model: tiny | base | small | medium | large
LANGUAGE   = "hi"      # source language of the audio
TASK       = "translate"  # "translate" → output in English; "transcribe" → keep source language

SUPPORTED_AUDIO_EXTENSIONS = (
    ".mp3",
    ".wav",
    ".m4a",
    ".flac",
    ".ogg",
    ".aac",
)


def extract_video_metadata(file_path):
    """
    Extract video number and title from the audio filename.

    Expected format:
        001_Introduction to Programming & Python | Python Tutorial.mp3

    Returns:
        video_number (int): 1
        title        (str): "Introduction to Programming & Python | Python Tutorial"
    """
    filename = os.path.basename(file_path)
    filename_without_extension = os.path.splitext(filename)[0]
    video_number_text, title = filename_without_extension.split("_", 1)
    video_number = int(video_number_text)
    return video_number, title


def create_chunks(result):
    """
    Convert Whisper segments directly into chunks.

    Each Whisper segment becomes one chunk.
    No sentence-based regrouping is performed here;
    merging happens in the next step (merge_chunks).
    """
    chunks = []
    print("Converting Whisper segments into chunks...")

    for chunk_number, segment in enumerate(result["segments"], start=1):
        chunk = {
            "chunk_number": chunk_number,
            "start": segment["start"],
            "end": segment["end"],
            "text": segment["text"].strip(),
        }
        chunks.append(chunk)
        print(
            f"  Chunk {chunk_number}: "
            f"{segment['start']:.2f}s -> {segment['end']:.2f}s"
        )

    return chunks


def process_audio(model, audio_path):
    """
    Transcribe one audio file with Whisper and save the result as JSON.
    Skips files that have already been processed.
    """
    filename = os.path.basename(audio_path)
    video_number, title = extract_video_metadata(audio_path)

    audio_name   = os.path.splitext(filename)[0]
    output_path  = os.path.join(OUTPUT_DIR, f"{audio_name}.json")

    # Skip already processed files
    if os.path.exists(output_path):
        print(f"\nSkipping: {filename}")
        print(f"JSON already exists: {output_path}")
        return

    print("\n" + "=" * 70)
    print(f"Processing: {filename}")
    print(f"Video number: {video_number}")
    print(f"Title: {title}")
    print("=" * 70)

    print("Whisper is transcribing and translating the audio...")
    result = model.transcribe(
        audio_path,
        language=LANGUAGE,
        task=TASK,
        word_timestamps=False,
    )
    print("Whisper completed.")

    chunks = create_chunks(result)

    output_data = {
        "video_number": video_number,
        "title": title,
        "chunks": chunks,
        "text": result["text"].strip(),
    }

    print("Saving JSON...")
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(output_data, file, ensure_ascii=False, indent=4)

    print(f"JSON saved: {output_path}")
    print(f"Total chunks: {len(chunks)}")


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Loading Whisper model: {MODEL_NAME}")
    model = whisper.load_model(MODEL_NAME)
    print("Whisper model loaded successfully.")

    audio_files = sorted(
        filename
        for filename in os.listdir(AUDIO_DIR)
        if filename.lower().endswith(SUPPORTED_AUDIO_EXTENSIONS)
    )

    print(f"\nFound {len(audio_files)} audio files.")

    for index, filename in enumerate(audio_files, start=1):
        print(f"\nFile {index}/{len(audio_files)}")
        audio_path = os.path.join(AUDIO_DIR, filename)
        try:
            process_audio(model, audio_path)
        except Exception as error:
            print(f"\nError processing: {filename}")
            print(f"Error: {error}")

    print("\n" + "=" * 70)
    print("All audio files have been processed.")
    print("=" * 70)


if __name__ == "__main__":
    main()


In [ ]:
import json
import os


# ── Configuration ──────────────────────────────────────────────────────────────
INPUT_FOLDER = "jsons"       # raw Whisper output produced by the cell above
OUTPUT_FOLDER = "main_jsons" # merged chunks (each chunk = GROUP_SIZE segments)
GROUP_SIZE = 5               # number of Whisper segments to merge into one chunk


def find_chunks_key(data):
    """Return the dict key whose value is a list of chunk dicts."""
    for key, value in data.items():
        if (
            isinstance(value, list)
            and value
            and isinstance(value[0], dict)
            and "chunk_number" in value[0]
        ):
            return key
    return None


def merge_chunks(chunks, group_size):
    """
    Merge every `group_size` consecutive Whisper segments into a single chunk.
    The merged chunk keeps the start time of the first segment and the end
    time of the last segment; texts are joined with a single space.
    """
    merged = []
    for i in range(0, len(chunks), group_size):
        group = chunks[i : i + group_size]

        new_chunk = dict(group[0])                          # copy metadata from first segment
        new_chunk["chunk_number"] = len(merged) + 1         # renumber: 1, 2, 3, …
        new_chunk["start"] = group[0]["start"]              # start of first segment
        new_chunk["end"] = group[-1]["end"]                  # end of last segment
        new_chunk["text"] = " ".join(c["text"].strip() for c in group)

        merged.append(new_chunk)
    return merged


def process_file(in_path, out_path):
    """Read one raw JSON file, merge its chunks, and write the result."""
    with open(in_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        key = find_chunks_key(data)
        if key is None:
            print(f"Skip (no chunks key found): {in_path}")
            return
        old_count = len(data[key])
        data[key] = merge_chunks(data[key], GROUP_SIZE)
        new_count = len(data[key])

    elif isinstance(data, list):
        # Flat list format: separate chunk dicts from any trailing metadata
        chunks = [c for c in data if "chunk_number" in c]
        others = [c for c in data if "chunk_number" not in c]
        old_count = len(chunks)
        data = merge_chunks(chunks, GROUP_SIZE) + others
        new_count = len(data) - len(others)
    else:
        print(f"Skip (unrecognised format): {in_path}")
        return

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"{os.path.basename(in_path)}: {old_count} segments -> {new_count} chunks")


def main():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    for name in sorted(os.listdir(INPUT_FOLDER)):
        if name.lower().endswith(".json"):
            process_file(
                os.path.join(INPUT_FOLDER, name),
                os.path.join(OUTPUT_FOLDER, name),
            )

    print("Done!")


if __name__ == "__main__":
    main()
